# 10 - Clock Drift QA

## Objetivo cientifico
Este notebook mide deriva de reloj (`sip_timestamp - participant_timestamp`) y jitter cuando el dataset contiene ambos relojes.

## Alcance real de este control
- Este control requiere dual-clock por evento: `participant_*` + `sip_*`.
- Si esa cobertura no existe, el resultado correcto no es PASS: es `NOT_APPLICABLE`.

## Politica operativa (practica)
- `APPLICABLE`: cuando existen ambas columnas de reloj y se puede calcular drift.
- `NOT_APPLICABLE`: cuando falta una o ambas columnas; el notebook documenta la limitacion y no fuerza una metrica inventada.

## Que aporta aunque hoy sea NOT_APPLICABLE
- Certifica trazablemente que la capacidad de QA de drift no esta disponible en el schema actual.
- Evita afirmaciones tecnicas no defendibles sobre latencia/clock-sync.
- Deja listo el control para activarse automaticamente cuando llegue feed dual-clock.

## Implicacion para estrategias
- Defendibles sin dual-clock: estrategias de barra/minuto y no latency-sensitive.
- No defendibles sin dual-clock: estrategias micro-latencia/event-time fino y claims de execution quality por reloj.

## Contexto regulatorio
- FINRA Rule 4590: https://www.finra.org/rules-guidance/rulebooks/finra-rules/4590
- CAT clock sync FAQ: https://www.catnmsplan.com/faq/r1



## Paso 1 - Setup y muestra controlada


In [1]:
NOTEBOOK_ID = "10_clock_drift_qa"

import json
import os
import sys
import uuid
import subprocess
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import polars as pl

try:
    import matplotlib.pyplot as plt
except Exception as e:
    plt = None
    print("matplotlib not available:", e)


def detect_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd] + list(cwd.parents)
    for cand in candidates:
        if (cand / "data" / "manifests").exists() and (cand / "notebooks" / "01_data_integrity").exists():
            return cand
    return cwd


PROJECT_ROOT = detect_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

DATA_ROOT = Path(os.getenv("DATA_CACHE_DIR", r"C:\TSIS_Data\data")).resolve()
SYMBOLS = ["AABA"]
MAX_FILES = 300
ROWS_PER_FILE = 2000

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:8]
OUT_DIR = PROJECT_ROOT / "runs" / "data_quality" / NOTEBOOK_ID / RUN_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)

try:
    git_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT).decode().strip()
except Exception:
    git_commit = "<unknown>"

print("Run ID:", RUN_ID)
print("Out dir:", OUT_DIR)
print("Data root:", DATA_ROOT)
print("Symbols:", SYMBOLS)


def list_trade_files(data_root: Path, symbols: list[str], max_files: int) -> list[Path]:
    files = []
    roots = ["trades_ticks_2004_2018", "trades_ticks_2019_2025"]
    for r in roots:
        for s in symbols:
            base = data_root / r / s
            if not base.exists():
                continue
            for fp in base.rglob("*.parquet"):
                files.append(fp)
                if len(files) >= max_files:
                    return files
    return files


def sample_concat(files: list[Path], rows_per_file: int) -> pl.DataFrame:
    parts = []
    for fp in files:
        try:
            df = pl.read_parquet(fp).head(rows_per_file)
            parts.append(df)
        except Exception:
            continue
    if not parts:
        return pl.DataFrame()
    return pl.concat(parts, how="diagonal_relaxed")



Run ID: 20260213_181527_74d5d23c
Out dir: C:\TSIS_Data\v1\backtest_SmallCaps\runs\data_quality\10_clock_drift_qa\20260213_181527_74d5d23c
Data root: C:\TSIS_Data\data
Symbols: ['AABA']


## Paso 2A - Schema gate (disponibilidad de relojes)


In [2]:
trade_files = list_trade_files(DATA_ROOT, SYMBOLS, MAX_FILES)
trades = sample_concat(trade_files, ROWS_PER_FILE)

pt_col = next((c for c in ["participant_timestamp", "participant_ts", "exchange_timestamp"] if c in trades.columns), None)
sip_col = next((c for c in ["sip_timestamp", "sip_ts"] if c in trades.columns), None)
venue_col = next((c for c in ["exchange", "exchange_id", "venue", "trf", "participant_id"] if c in trades.columns), None)

ts_col = next((c for c in ["timestamp", "t", "ts"] if c in trades.columns), None)

print("rows:", trades.height)
print("columns:", sorted(trades.columns))
print("participant_ts:", pt_col, "sip_ts:", sip_col, "venue:", venue_col, "timestamp:", ts_col)

schema_gate = pl.DataFrame([
    {
        "rows_sampled": int(trades.height),
        "trade_files_sampled": int(len(trade_files)),
        "participant_ts_col": pt_col,
        "sip_ts_col": sip_col,
        "venue_col": venue_col,
        "timestamp_col": ts_col,
        "clock_drift_check_status": "available" if (pt_col is not None and sip_col is not None) else "not_available",
    }
])

schema_gate



rows: 298791
columns: ['conditions', 'date', 'exchange', 'price', 'size', 'ticker', 'timestamp']
participant_ts: None sip_ts: None venue: exchange timestamp: timestamp


rows_sampled,trade_files_sampled,participant_ts_col,sip_ts_col,venue_col,timestamp_col,clock_drift_check_status
i64,i64,null,null,str,str,str
298791,300,null,null,"""exchange""","""timestamp""","""not_available"""


## Paso 2B - Metricas globales de drift y jitter


In [3]:
if trades.height == 0 or pt_col is None or sip_col is None:
    drift_stats = pl.DataFrame()
    global_metrics = pl.DataFrame([
        {
            "n_rows_with_clocks": 0,
            "drift_mean_ms": None,
            "drift_std_ms": None,
            "drift_p99_ms": None,
            "abs_drift_p99_ms": None,
            "clock_drift_check_status": "not_available",
        }
    ])
else:
    d = (
        trades.with_columns([
            pl.col(pt_col).cast(pl.Int64, strict=False),
            pl.col(sip_col).cast(pl.Int64, strict=False),
        ])
        .drop_nulls([pt_col, sip_col])
        .with_columns([
            ((pl.col(sip_col) - pl.col(pt_col)) / 1_000_000.0).alias("drift_ms"),
            pl.when(venue_col is not None).then(pl.col(venue_col).cast(pl.Utf8)).otherwise(pl.lit("UNKNOWN")).alias("venue_key"),
        ])
    )

    drift_stats = (
        d.group_by("venue_key")
        .agg([
            pl.len().alias("n_rows"),
            pl.col("drift_ms").mean().alias("drift_mean_ms"),
            pl.col("drift_ms").std().alias("drift_std_ms"),
            pl.col("drift_ms").quantile(0.99).alias("drift_p99_ms"),
            pl.col("drift_ms").abs().quantile(0.99).alias("abs_drift_p99_ms"),
        ])
        .sort("n_rows", descending=True)
    )

    gm = d.select([
        pl.len().alias("n_rows_with_clocks"),
        pl.col("drift_ms").mean().alias("drift_mean_ms"),
        pl.col("drift_ms").std().alias("drift_std_ms"),
        pl.col("drift_ms").quantile(0.99).alias("drift_p99_ms"),
        pl.col("drift_ms").abs().quantile(0.99).alias("abs_drift_p99_ms"),
    ])
    global_metrics = gm.with_columns(pl.lit("available").alias("clock_drift_check_status"))

print("Global drift metrics:")
print(global_metrics)

global_metrics



Global drift metrics:
shape: (1, 6)
┌─────────────────┬───────────────┬──────────────┬──────────────┬─────────────────┬────────────────┐
│ n_rows_with_clo ┆ drift_mean_ms ┆ drift_std_ms ┆ drift_p99_ms ┆ abs_drift_p99_m ┆ clock_drift_ch │
│ cks             ┆ ---           ┆ ---          ┆ ---          ┆ s               ┆ eck_status     │
│ ---             ┆ null          ┆ null         ┆ null         ┆ ---             ┆ ---            │
│ i64             ┆               ┆              ┆              ┆ null            ┆ str            │
╞═════════════════╪═══════════════╪══════════════╪══════════════╪═════════════════╪════════════════╡
│ 0               ┆ null          ┆ null         ┆ null         ┆ null            ┆ not_available  │
└─────────────────┴───────────────┴──────────────┴──────────────┴─────────────────┴────────────────┘


n_rows_with_clocks,drift_mean_ms,drift_std_ms,drift_p99_ms,abs_drift_p99_ms,clock_drift_check_status
i64,null,null,null,null,str
0,null,null,null,null,"""not_available"""


## Paso 2C - Drill-down por venue


In [4]:
if drift_stats.height > 0:
    print("Top venue_key by abs_drift_p99_ms:")
    print(drift_stats.select(["venue_key", "n_rows", "drift_mean_ms", "drift_std_ms", "drift_p99_ms", "abs_drift_p99_ms"]).sort("abs_drift_p99_ms", descending=True).head(15))
else:
    print("No drift stats computed: participant/sip clocks not available in sampled schema.")

drift_stats



No drift stats computed: participant/sip clocks not available in sampled schema.


shape: (0, 0)
┌┐
╞╡
└┘

## Paso 3A - Visual de drift p99 por venue


In [5]:
if plt is not None and drift_stats.height > 0:
    p = pd.DataFrame(drift_stats.head(12).to_dicts())
    plt.style.use("seaborn-v0_8-darkgrid")
    plt.figure(figsize=(12, 4))
    plt.bar(p["venue_key"].astype(str), p["abs_drift_p99_ms"])
    plt.title("Absolute clock drift p99 by venue-like key")
    plt.ylabel("abs_drift_p99_ms")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("No drift plot: missing participant/sip clock columns in sampled data.")



No drift plot: missing participant/sip clock columns in sampled data.


## Paso 4 - Calibracion automatica y decision de aplicabilidad
Si no hay dual-clock, el resultado final es `NOT_APPLICABLE` y se emiten acciones recomendadas.


In [6]:
runs_root = PROJECT_ROOT / "runs" / "data_quality" / NOTEBOOK_ID


def _quantile(vals: list[float], q: float) -> float:
    if not vals:
        return float("nan")
    x = sorted(float(v) for v in vals)
    if len(x) == 1:
        return x[0]
    pos = (len(x) - 1) * q
    lo = int(pos)
    hi = min(lo + 1, len(x) - 1)
    frac = pos - lo
    return x[lo] * (1.0 - frac) + x[hi] * frac


def _calibrate(vals: list[float], default_pass: float, default_warn: float) -> dict:
    vals = [float(v) for v in vals if v is not None]
    n = len(vals)
    if n >= 20:
        p_pass = _quantile(vals, 0.95)
        p_warn = _quantile(vals, 0.99)
        mode = "hist_p95_p99"
    elif n >= 5:
        p_pass = _quantile(vals, 0.90)
        p_warn = _quantile(vals, 0.98)
        mode = "hist_p90_p98"
    elif n > 0:
        p_pass = max(vals)
        p_warn = max(vals) * 1.25
        mode = "hist_max_buffer"
    else:
        p_pass = default_pass
        p_warn = default_warn
        mode = "default_fallback"

    p_pass = max(0.0, float(p_pass))
    p_warn = max(p_pass + 1e-9, float(p_warn))
    return {"n_history": n, "mode": mode, "pass_max": p_pass, "warn_max": p_warn}


def _status(value, pass_max: float, warn_max: float):
    if value is None:
        return "WARN", "metric_not_available"
    v = float(value)
    if v <= pass_max:
        return "PASS", "within_pass_threshold"
    if v <= warn_max:
        return "WARN", "within_warn_threshold"
    return "FAIL", "exceeds_warn_threshold"


metrics = schema_gate.hstack(global_metrics.select([
    "n_rows_with_clocks",
    "drift_mean_ms",
    "drift_std_ms",
    "drift_p99_ms",
    "abs_drift_p99_ms",
]))

mrow = metrics.to_dicts()[0] if metrics.height > 0 else {}
has_dual_clock = mrow.get("clock_drift_check_status") == "available"
applicability_status = "APPLICABLE" if has_dual_clock else "NOT_APPLICABLE"

hist_abs_p99 = []
if runs_root.exists():
    for d in sorted([x for x in runs_root.iterdir() if x.is_dir()], key=lambda x: x.stat().st_mtime):
        if d.name == RUN_ID:
            continue
        m_path = d / "clock_drift_metrics.parquet"
        if m_path.exists():
            try:
                hm = pl.read_parquet(m_path)
                if hm.height > 0:
                    row = hm.to_dicts()[0]
                    v = row.get("abs_drift_p99_ms")
                    if v is not None:
                        hist_abs_p99.append(float(v))
            except Exception:
                pass

thr = _calibrate(hist_abs_p99, default_pass=50.0, default_warn=200.0)

if applicability_status == "NOT_APPLICABLE":
    gate_breakdown = [
        {
            "gate": "clock_schema_coverage",
            "status": "NOT_APPLICABLE",
            "reason": "participant_or_sip_clock_missing",
        },
        {
            "gate": "abs_drift_p99_ms",
            "status": "NOT_APPLICABLE",
            "reason": "metric_not_available_without_dual_clock",
            "value": None,
            "pass_max": thr["pass_max"],
            "warn_max": thr["warn_max"],
            "history_n": thr["n_history"],
            "mode": thr["mode"],
        },
    ]
    overall_status = "NOT_APPLICABLE"
else:
    drift_status, drift_reason = _status(mrow.get("abs_drift_p99_ms"), thr["pass_max"], thr["warn_max"])
    gate_breakdown = [
        {"gate": "clock_schema_coverage", "status": "PASS", "reason": "clock_columns_available"},
        {
            "gate": "abs_drift_p99_ms",
            "status": drift_status,
            "reason": drift_reason,
            "value": mrow.get("abs_drift_p99_ms"),
            "pass_max": thr["pass_max"],
            "warn_max": thr["warn_max"],
            "history_n": thr["n_history"],
            "mode": thr["mode"],
        },
    ]

    statuses = [g["status"] for g in gate_breakdown]
    if "FAIL" in statuses:
        overall_status = "FAIL"
    elif "WARN" in statuses:
        overall_status = "WARN"
    else:
        overall_status = "PASS"

calibration = {
    "global_abs_drift_p99_thresholds": thr,
}

recommended_actions = []
if applicability_status == "NOT_APPLICABLE":
    recommended_actions.append({
        "priority": "high",
        "action": "ingest_dual_clock_dataset",
        "detail": "Incorporar feed con participant_timestamp y sip_timestamp para habilitar clock-drift QA real.",
    })
    recommended_actions.append({
        "priority": "medium",
        "action": "set_strategy_scope",
        "detail": "Limitar estrategias a horizontes no latency-sensitive mientras no exista dual-clock.",
    })
elif overall_status == "FAIL":
    recommended_actions.append({
        "priority": "high",
        "action": "block_latency_sensitive_production",
        "detail": "No promover a produccion estrategias sensibles a latencia hasta corregir drift.",
    })

print("Applicability:", applicability_status)
print("Calibration:", calibration)
print("Gate breakdown:", gate_breakdown)
print("OVERALL_STATUS:", overall_status)
print("Recommended actions:")
print(recommended_actions)



Applicability: NOT_APPLICABLE
Calibration: {'global_abs_drift_p99_thresholds': {'n_history': 0, 'mode': 'default_fallback', 'pass_max': 50.0, 'warn_max': 200.0}}
Gate breakdown: [{'gate': 'clock_schema_coverage', 'status': 'NOT_APPLICABLE', 'reason': 'participant_or_sip_clock_missing'}, {'gate': 'abs_drift_p99_ms', 'status': 'NOT_APPLICABLE', 'reason': 'metric_not_available_without_dual_clock', 'value': None, 'pass_max': 50.0, 'warn_max': 200.0, 'history_n': 0, 'mode': 'default_fallback'}]
OVERALL_STATUS: NOT_APPLICABLE
Recommended actions:
[{'priority': 'high', 'action': 'ingest_dual_clock_dataset', 'detail': 'Incorporar feed con participant_timestamp y sip_timestamp para habilitar clock-drift QA real.'}, {'priority': 'medium', 'action': 'set_strategy_scope', 'detail': 'Limitar estrategias a horizontes no latency-sensitive mientras no exista dual-clock.'}]


## Paso 5 - Export y decision operativa


In [7]:
if metrics.height > 0:
    metrics.write_parquet(OUT_DIR / "clock_drift_metrics.parquet")
if drift_stats.height > 0:
    drift_stats.write_parquet(OUT_DIR / "clock_drift_stats_by_venue.parquet")

with open(OUT_DIR / "clock_drift_calibration.json", "w", encoding="utf-8") as f:
    json.dump(calibration, f, indent=2)

statuses = [g.get("status", "PASS") for g in gate_breakdown]
if applicability_status == "NOT_APPLICABLE":
    root_cause = "schema_gap"
elif any(s in {"FAIL", "WARN"} for s in statuses):
    root_cause = "microstructure_friction"
else:
    root_cause = "none"

decision_table = [{
    "ticker": SYMBOLS[0] if SYMBOLS else "<unknown>",
    "applicability_status": applicability_status,
    "overall_status": overall_status,
    "root_cause": root_cause,
    "decision": overall_status,
}]

decision = {
    "check_id": "clock_drift_qa_v3",
    "as_of_utc": datetime.now(timezone.utc).isoformat(),
    "git_commit": git_commit,
    "symbols": SYMBOLS,
    "applicability_status": applicability_status,
    "overall_status": overall_status,
    "root_cause": root_cause,
    "metrics": metrics.to_dicts(),
    "gate_breakdown": gate_breakdown,
    "decision_table": decision_table,
    "calibration": calibration,
    "recommended_actions": recommended_actions,
    "strategy_scope_policy": {
        "defensible_without_dual_clock": [
            "bar_minute_or_higher_frequency_aggregation",
            "non_latency_sensitive_execution_assumptions"
        ],
        "not_defensible_without_dual_clock": [
            "latency_sensitive_event_driven_alpha",
            "clock_sync_claims_between_participant_and_sip"
        ]
    }
}

with open(OUT_DIR / "clock_drift_decision.json", "w", encoding="utf-8") as f:
    json.dump(decision, f, indent=2)

print("Saved:", OUT_DIR / "clock_drift_metrics.parquet")
print("Saved:", OUT_DIR / "clock_drift_stats_by_venue.parquet")
print("Saved:", OUT_DIR / "clock_drift_calibration.json")
print("Saved:", OUT_DIR / "clock_drift_decision.json")
print("APPLICABILITY_STATUS:", applicability_status)
print("OVERALL_STATUS:", overall_status)



Saved: C:\TSIS_Data\v1\backtest_SmallCaps\runs\data_quality\10_clock_drift_qa\20260213_181527_74d5d23c\clock_drift_metrics.parquet
Saved: C:\TSIS_Data\v1\backtest_SmallCaps\runs\data_quality\10_clock_drift_qa\20260213_181527_74d5d23c\clock_drift_stats_by_venue.parquet
Saved: C:\TSIS_Data\v1\backtest_SmallCaps\runs\data_quality\10_clock_drift_qa\20260213_181527_74d5d23c\clock_drift_calibration.json
Saved: C:\TSIS_Data\v1\backtest_SmallCaps\runs\data_quality\10_clock_drift_qa\20260213_181527_74d5d23c\clock_drift_decision.json
APPLICABILITY_STATUS: NOT_APPLICABLE
OVERALL_STATUS: NOT_APPLICABLE
